In [1]:
import os, sys, re, json, hashlib
from pathlib import Path
from datetime import datetime

NORTHSTAR = "primes-t11-qa"
NOTEBOOK_PATH = "notebooks/qa/41-primes-t11-qa.ipynb"
TOWER = 11
TOWER_NAME = "Tower of Primes"
MIN_SCORE = 70

BROWSER_ROOT = Path("/home/phuc/projects/solace-browser")
SRC = BROWSER_ROOT / "src"
WEB = BROWSER_ROOT / "web"
TESTS = BROWSER_ROOT / "tests"

results = []

def record(probe_id, name, passed, detail=""):
    status = "PASS" if passed else "FAIL"
    results.append({"id": probe_id, "name": name, "status": status, "detail": detail})
    print(f"{status}: {name}" + (f" -- {detail}" if detail else ""))

def read_file(path):
    p = Path(path)
    return p.read_text(encoding='utf-8', errors='replace') if p.exists() else ""

print(f"Tower {TOWER}: {TOWER_NAME}")
assert BROWSER_ROOT.exists()

Tower 11: Tower of Primes


In [2]:
# Cell 1: Code Quality - No eval() or document.write() in JS files (Floors 1-7)

js_dir = WEB / "js"
js_files = list(js_dir.glob("*.js")) if js_dir.exists() else []

# P1: No eval() calls in JS
eval_hits = []
for f in js_files:
    content = read_file(f)
    # Match eval( but not .evaluate or similar method names
    matches = re.findall(r'(?<!\w)eval\s*\(', content)
    if matches:
        eval_hits.append(f.name)
record("P1", "No eval() in JS files", len(eval_hits) == 0,
       f"Found eval() in: {eval_hits}" if eval_hits else f"Checked {len(js_files)} JS files")

# P2: No document.write() in JS
docwrite_hits = []
for f in js_files:
    content = read_file(f)
    if 'document.write(' in content:
        docwrite_hits.append(f.name)
record("P2", "No document.write() in JS files", len(docwrite_hits) == 0,
       f"Found document.write in: {docwrite_hits}" if docwrite_hits else f"Checked {len(js_files)} JS files")

# P3: No eval() in HTML files (inline scripts)
html_files = list(WEB.glob("*.html")) if WEB.exists() else []
eval_html = []
for f in html_files:
    content = read_file(f)
    # Look for eval( in script sections only
    scripts = re.findall(r'<script[^>]*>(.*?)</script>', content, re.DOTALL)
    for script in scripts:
        if re.search(r'(?<!\w)eval\s*\(', script):
            eval_html.append(f.name)
            break
record("P3", "No eval() in HTML inline scripts", len(eval_html) == 0,
       f"Found eval() in: {eval_html}" if eval_html else f"Checked {len(html_files)} HTML files")

PASS: No eval() in JS files -- Checked 16 JS files
PASS: No document.write() in JS files -- Checked 16 JS files
PASS: No eval() in HTML inline scripts -- Checked 20 HTML files


In [3]:
# Cell 2: CSS Design Token System (Floors 8-15)

site_css = read_file(WEB / "css" / "site.css")

# P4: Design tokens --sb-* defined in site.css
sb_vars = re.findall(r'--sb-[a-z0-9-]+', site_css)
unique_sb = set(sb_vars)
record("P4", "CSS design tokens --sb-* defined", len(unique_sb) >= 10,
       f"Found {len(unique_sb)} unique --sb-* tokens")

# P5: Font size design tokens defined
font_tokens = re.findall(r'--font-[a-z0-9]+', site_css)
unique_fonts = set(font_tokens)
record("P5", "Font size design tokens defined (--font-*)", len(unique_fonts) >= 5,
       f"Found {len(unique_fonts)} font tokens: {sorted(unique_fonts)[:6]}")

# P6: Theme CSS files exist (dark, light, midnight)
theme_dir = WEB / "css" / "themes"
expected_themes = ["dark.css", "light.css", "midnight.css"]
themes_found = [t for t in expected_themes if (theme_dir / t).exists()]
record("P6", "Theme CSS files exist (dark/light/midnight)", len(themes_found) == 3,
       f"Found: {themes_found}")

PASS: CSS design tokens --sb-* defined -- Found 78 unique --sb-* tokens
PASS: Font size design tokens defined (--font-*) -- Found 10 font tokens: ['--font-2xl', '--font-3xl', '--font-4xl', '--font-5xl', '--font-base', '--font-lg']
PASS: Theme CSS files exist (dark/light/midnight) -- Found: ['dark.css', 'light.css', 'midnight.css']


In [4]:
# Cell 3: Test Suite and Evidence Hashing (Floors 16-24)

# P7: Test directory exists with test files
test_files = list(TESTS.glob("test_*.py")) if TESTS.exists() else []
record("P7", "Test suite exists with test files", len(test_files) >= 10,
       f"Found {len(test_files)} test files in tests/")

# P8: Evidence module uses SHA-256
evidence_init = read_file(SRC / "evidence" / "__init__.py")
has_sha256 = "sha256" in evidence_init.lower() and "hashlib" in evidence_init
record("P8", "Evidence module uses SHA-256 hashing", has_sha256,
       "hashlib.sha256 found in src/evidence/__init__.py")

# P9: Evidence uses sort_keys=True for deterministic JSON
has_sort_keys = "sort_keys=True" in evidence_init
record("P9", "Evidence uses sort_keys=True for deterministic hashing", has_sort_keys,
       "json.dumps(sort_keys=True) found in evidence module")

# P10: Evidence chain has verify_chain function
has_verify = "def verify_chain" in evidence_init
record("P10", "Evidence chain has verify_chain function", has_verify,
       "verify_chain() function exists in evidence/__init__.py")

# P11: Evidence chain has seal_evidence function
has_seal = "def seal_evidence" in evidence_init
record("P11", "Evidence chain has seal_evidence function", has_seal,
       "seal_evidence() function exists in evidence/__init__.py")

PASS: Test suite exists with test files -- Found 107 test files in tests/
PASS: Evidence module uses SHA-256 hashing -- hashlib.sha256 found in src/evidence/__init__.py
PASS: Evidence uses sort_keys=True for deterministic hashing -- json.dumps(sort_keys=True) found in evidence module
PASS: Evidence chain has verify_chain function -- verify_chain() function exists in evidence/__init__.py
PASS: Evidence chain has seal_evidence function -- seal_evidence() function exists in evidence/__init__.py


In [5]:
# Cell 4: Fallback Ban Compliance (Floors 25-32)

# P12: No bare except Exception in src/oauth3/
oauth3_dir = SRC / "oauth3"
oauth3_files = list(oauth3_dir.glob("*.py")) if oauth3_dir.exists() else []
bare_except_files = []
for f in oauth3_files:
    content = read_file(f)
    # Match 'except Exception:' with optional whitespace (no specific catch)
    if re.search(r'except\s+Exception\s*:', content):
        bare_except_files.append(f.name)
record("P12", "No bare 'except Exception:' in oauth3 module", len(bare_except_files) == 0,
       f"Violations: {bare_except_files}" if bare_except_files else f"Checked {len(oauth3_files)} oauth3 files")

# P13: No bare except Exception in evidence module (excluding comments/docstrings)
evidence_dir = SRC / "evidence"
evidence_files = list(evidence_dir.glob("*.py")) if evidence_dir.exists() else []
bare_evidence = []
for f in evidence_files:
    content = read_file(f)
    # Check line by line — only count actual code lines, not comments or docstrings
    in_docstring = False
    for line in content.splitlines():
        stripped = line.strip()
        # Track triple-quote docstrings
        if '"""' in stripped or "'''" in stripped:
            quote = '"""' if '"""' in stripped else "'''"
            count = stripped.count(quote)
            if count == 1:
                in_docstring = not in_docstring
            # If count >= 2 on same line, it's a single-line docstring — not entering
            continue
        if in_docstring:
            continue
        # Skip comment lines
        if stripped.startswith("#"):
            continue
        if re.search(r'except\s+Exception\s*:', stripped):
            bare_evidence.append(f.name)
            break
record("P13", "No bare 'except Exception:' in evidence module (code only)", len(bare_evidence) == 0,
       f"Violations: {bare_evidence}" if bare_evidence else f"Checked {len(evidence_files)} evidence files (comments excluded)")

# P14: No 'except Exception: pass' pattern across web/js/
# In JS, the equivalent is catch(e) {} or catch {} with empty body
# But since this is JS, we check for empty catch blocks
empty_catch_files = []
for f in js_files:
    content = read_file(f)
    # Very loose match: catch with empty or whitespace-only body
    matches = re.findall(r'catch\s*\([^)]*\)\s*\{\s*\}', content)
    if matches:
        empty_catch_files.append(f.name)
record("P14", "No empty catch blocks in JS files", len(empty_catch_files) == 0,
       f"Violations: {empty_catch_files}" if empty_catch_files else f"Checked {len(js_files)} JS files")

PASS: No bare 'except Exception:' in oauth3 module -- Checked 9 oauth3 files
PASS: No bare 'except Exception:' in evidence module (code only) -- Checked 2 evidence files (comments excluded)
PASS: No empty catch blocks in JS files -- Checked 16 JS files


In [6]:
# Cell 5: Lint/Format Config and Build Reproducibility (Floors 33-40)

# P15: Ruff linter configured in pyproject.toml
pyproject = read_file(BROWSER_ROOT / "pyproject.toml")
has_ruff = "[tool.ruff]" in pyproject
record("P15", "Ruff linter configured in pyproject.toml", has_ruff,
       "[tool.ruff] section found")

# P16: Pytest configured in pyproject.toml
has_pytest = "[tool.pytest" in pyproject
record("P16", "Pytest configured in pyproject.toml", has_pytest,
       "[tool.pytest.ini_options] section found")

# P17: package.json has test script
pkg_json = read_file(BROWSER_ROOT / "package.json")
pkg_data = json.loads(pkg_json) if pkg_json else {}
has_test_script = "test" in pkg_data.get("scripts", {})
record("P17", "package.json has test script", has_test_script,
       f"test script: {pkg_data.get('scripts', {}).get('test', 'N/A')[:60]}")

# P18: package-lock.json exists (dependency lockfile)
has_lockfile = (BROWSER_ROOT / "package-lock.json").exists()
record("P18", "package-lock.json exists (dependency lockfile)", has_lockfile,
       "Dependency versions are locked")

# P19: ESLint or Prettier configured as devDependency
dev_deps = pkg_data.get("devDependencies", {})
has_lint_tool = "eslint" in dev_deps or "prettier" in dev_deps
record("P19", "ESLint or Prettier in devDependencies", has_lint_tool,
       f"Found: {[k for k in dev_deps if k in ('eslint', 'prettier')]}")

PASS: Ruff linter configured in pyproject.toml -- [tool.ruff] section found
PASS: Pytest configured in pyproject.toml -- [tool.pytest.ini_options] section found
PASS: package.json has test script -- test script: python -m pytest tests/ -x -q --tb=short -m 'not integration
PASS: package-lock.json exists (dependency lockfile) -- Dependency versions are locked
PASS: ESLint or Prettier in devDependencies -- Found: ['eslint', 'prettier']


In [7]:
# Cell 6: Build Reproducibility and Integrity (Floors 41-49)

# P20: CSS integrity file exists (site.css.sha256)
css_sha = (WEB / "css" / "site.css.sha256").exists()
record("P20", "CSS integrity file exists (site.css.sha256)", css_sha,
       "SHA-256 hash of site.css is tracked")

# P21: Evidence module has reset_chain function (for test isolation)
has_reset = "def reset_chain" in evidence_init
record("P21", "Evidence module has reset_chain for test isolation", has_reset,
       "reset_chain() function exists in evidence/__init__.py")

# P22: Evidence genesis hash is defined
has_genesis = 'GENESIS_HASH' in evidence_init
record("P22", "Evidence GENESIS_HASH constant defined", has_genesis,
       "GENESIS_HASH used as chain anchor")

# P23: 47 locale files exist (STORY-47 prime)
locale_dir = BROWSER_ROOT / "app" / "locales" / "yinyang"
locale_files = list(locale_dir.glob("*.json")) if locale_dir.exists() else []
record("P23", "Exactly 47 locale files (STORY-47 prime)", len(locale_files) == 47,
       f"Found {len(locale_files)} locale files")

PASS: CSS integrity file exists (site.css.sha256) -- SHA-256 hash of site.css is tracked
PASS: Evidence module has reset_chain for test isolation -- reset_chain() function exists in evidence/__init__.py
PASS: Evidence GENESIS_HASH constant defined -- GENESIS_HASH used as chain anchor
PASS: Exactly 47 locale files (STORY-47 prime) -- Found 47 locale files


In [8]:
passed = len([r for r in results if r["status"] == "PASS"])
failed = len([r for r in results if r["status"] == "FAIL"])
total = len(results)
score = round(passed / total * 100, 1) if total > 0 else 0
finding_rate = round(failed / total * 100, 1) if total > 0 else 0
grade = "A+" if score >= 95 else "A" if score >= 90 else "B+" if score >= 85 else "B" if score >= 80 else "C" if score >= 70 else "D" if score >= 60 else "F"
summary = {"surface": NORTHSTAR, "tower": TOWER, "tower_name": TOWER_NAME, "timestamp": datetime.now().isoformat(), "total_probes": total, "passed": passed, "failed": failed, "score": score, "grade": grade, "finding_rate": finding_rate, "converged": finding_rate < 5, "findings": [r for r in results if r["status"] == "FAIL"], "evidence_hash": hashlib.sha256(json.dumps(results, sort_keys=True).encode()).hexdigest()[:16]}
print("=" * 60)
print(f"TOWER {TOWER}: {TOWER_NAME}")
print("=" * 60)
print(f"Probes: {total} | Passed: {passed} | Failed: {failed}")
print(f"Score: {score}% | Grade: {grade} | Finding rate: {finding_rate}%")
print(f"Converged: {'YES' if summary['converged'] else 'NO'}")
print(f"Evidence hash: {summary['evidence_hash']}")
if summary["findings"]:
    print("\nFINDINGS:")
    for f in summary["findings"]:
        print(f"  [{f['id']}] {f['name']}: {f.get('detail', '')}")
print()
print(json.dumps(summary, indent=2))

TOWER 11: Tower of Primes
Probes: 23 | Passed: 23 | Failed: 0
Score: 100.0% | Grade: A+ | Finding rate: 0.0%
Converged: YES
Evidence hash: 3952fecab482833f

{
  "surface": "primes-t11-qa",
  "tower": 11,
  "tower_name": "Tower of Primes",
  "timestamp": "2026-03-06T11:33:53.563048",
  "total_probes": 23,
  "passed": 23,
  "failed": 0,
  "score": 100.0,
  "grade": "A+",
  "finding_rate": 0.0,
  "converged": true,
  "findings": [],
  "evidence_hash": "3952fecab482833f"
}
